In [ ]:
# ── Step 0: Kill any other active Colab sessions for this account ──────────
try:
    from google.colab import _message
    sessions = _message.blocking_request('get_sessions', request='', timeout_sec=5)
    if sessions and 'sessions' in sessions:
        current = sessions.get('currentSessionId', '')
        for s in sessions['sessions']:
            sid = s.get('sessionId', '')
            if sid and sid != current:
                try:
                    _message.blocking_request('terminate_session', request={'sessionId': sid}, timeout_sec=5)
                    print(f'✅ Killed old session: {sid[:12]}...')
                except Exception as e:
                    print(f'⚠️  Could not kill session {sid[:12]}: {e}')
except Exception as e:
    print(f'Session cleanup skipped: {e}')

# ── Step 1: GPU Check — auto-reconnect if no GPU ───────────────────────────
import subprocess, sys

def check_gpu():
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                                capture_output=True, text=True, timeout=10)
        if result.returncode == 0 and result.stdout.strip():
            print(f'✅ GPU Ready: {result.stdout.strip()}')
            return True
    except Exception:
        pass
    return False

if not check_gpu():
    print('❌ No GPU detected! Switching to GPU runtime automatically...')
    try:
        from google.colab import runtime
        runtime.unassign()  # Forces reconnect — Colab will reassign with GPU
    except Exception as e:
        print(f'Please manually go to Runtime → Change runtime type → T4 GPU')
        raise SystemExit('GPU required. Please enable GPU and re-run.')

# ── Step 2: Kill any leftover cloudflared or ComfyUI from previous run ─────
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
print('✅ Cleaned up old processes')

# ── Step 3: Install ComfyUI ────────────────────────────────────────────────
import os
if not os.path.exists('/content/ComfyUI'):
    print('Installing ComfyUI...')
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
    %cd ComfyUI
    !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
    !pip install -q -r requirements.txt
else:
    print('✅ ComfyUI already installed')
    %cd /content/ComfyUI

# ── Step 4: Download models (skip if already exist) ────────────────────────
import os
models = [
    ('models/checkpoints/sd_xl_base_1.0.safetensors',
     'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'),
    ('models/upscale_models/RealESRGAN_x4plus.pth',
     'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'),
    ('models/vae/sdxl_vae.safetensors',
     'https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors'),
]
for fpath, url in models:
    if not os.path.exists(fpath):
        print(f'Downloading {os.path.basename(fpath)}...')
        !wget -q -O {fpath} {url}
    else:
        print(f'✅ {os.path.basename(fpath)} already exists')

# ── Step 5: Install Cloudflared ────────────────────────────────────────────
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
print('✅ Cloudflared ready')

# ── Step 6: Cloudflare tunnel + URL reporter ───────────────────────────────
import re, threading, time, socket
from IPython.display import display, HTML

def iframe_thread(port):
    # Wait for ComfyUI to be ready
    for _ in range(120):
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            sock.close()
            break
        sock.close()
    print('\nComfyUI is running. Starting Cloudflare tunnel...')

    p = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in p.stdout:
        l = line.decode(errors='ignore')
        m = re.search(r'https?://[a-zA-Z0-9-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print('\n=======================================================')
            print(f'\u2705 YOUR SERVER URL: {url}')
            display(HTML(f'<a href="{url}" target="_blank" style="font-size:1.2em;font-weight:bold">{url}</a>'))
            print('=======================================================\n')
            break

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ── Step 7: Launch ComfyUI ─────────────────────────────────────────────────
print('\n🚀 Starting ComfyUI...')
!python main.py --dont-print-server --enable-cors-header "*"